In [1]:
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
import undetected_chromedriver as uc
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
from datetime import datetime
import time
import re
import pandas as pd

In [2]:
driver = uc.Chrome(version_main=148)
url = "https://batdongsan.com.vn/ban-can-ho-chung-cu-ha-noi?vrs=1&cIds=650,362"
driver.get(url)
time.sleep(3)

In [32]:
_original_del = uc.Chrome.__del__
def _safe_del(self):
    try:
        _original_del(self)
    except Exception:
        pass
uc.Chrome.__del__ = _safe_del

In [3]:
from bs4 import BeautifulSoup
import undetected_chromedriver as uc


def Get_urls(log_file="crawl_log.txt"):
    driver = uc.Chrome(version_main=148)
    all_urls = set()

    try:
        with open(log_file, "w", encoding="utf-8") as f:

            for page_num in range(1, 51):

                if page_num == 1:
                    url = "https://batdongsan.com.vn/ban-can-ho-chung-cu-ha-noi?vrs=1&cIds=650,362"
                else:
                    url = f"https://batdongsan.com.vn/ban-can-ho-chung-cu-ha-noi/p{page_num}?cIds=650,362&vrs=1"

                driver.get(url)

                soup = BeautifulSoup(driver.page_source, "html.parser")
                links = soup.find_all('a', class_='js__product-link-for-product-id')

                page_urls = set()

                for i in links:
                    href = i.get('href')
                    if not href:
                        continue

                    if "vaymuanha.batdongsan.com.vn" in href:
                        continue

                    full_url = "https://batdongsan.com.vn" + href

                    all_urls.add(full_url)
                    page_urls.add(full_url)

                log_line = f"Page {page_num}: {len(page_urls)} links\n"
                f.write(log_line)

            f.write(f"\nTOTAL: {len(all_urls)} links\n")

    finally:
        driver.quit()

    return list(all_urls)

In [4]:
all_urls = Get_urls()

In [73]:
print(len(all_urls))

967


In [74]:
print(all_urls[0])

https://batdongsan.com.vn/ban-nha-mat-pho-duong-tran-quoc-vuong-phuong-dich-vong-hau-7/toa-khach-san-105m2-8t-mt-10-5m-lo-goc-via-he-rong-dong-tien-4-ty-nam-pr44964531


In [ ]:
driver.get(all_urls[0])
page_source = BeautifulSoup(driver.page_source,'html.parser')
print(page_source)

In [86]:
driver.get(all_urls[0])
time.sleep(2)
page_source = BeautifulSoup(driver.page_source, 'html.parser')

# Tìm code
info_div1 = page_source.find('div', class_='re__pr-short-info re__pr-config js__pr-config')
value_text_code = "NAN"
if info_div1:
    all_items = info_div1.find_all('div', class_='re__pr-short-info-item js__pr-config-item')
    item = all_items[3] if len(all_items) > 3 else None
    if item:
        value = item.find('span', class_='value')
        value_text_code = value.get_text().strip() if value else "NAN"

# Tìm title
info_div_title = page_source.find('div', class_='re__breadcrumb js__breadcrumb js__ob-breadcrumb')
value_text_title = "NAN"
if info_div_title:
    all_items3 = info_div_title.find_all('a', class_='re__link-se')
    item1 = all_items3[3] if len(all_items3) > 3 else None
    value_text_title= item1.get_text().strip() if item1 else "NAN"

# Địa chỉ
parent_div = page_source.find('div', class_='re__ldp-address')
Address = parent_div.find('span', class_='re__address') if parent_div else None
address = parent_div.find('span', class_='re__address-line-2') if parent_div else None
address_text = address.get_text(strip=True) if address else "NAN"

# Tìm date
info_div2 = page_source.find('div', class_='re__pr-short-info re__pr-config js__pr-config')
value_text_date = "NAN"
if info_div2:
    ngay = info_div2.find('div')
    value_text_date = ngay.find('span', class_='value').get_text().strip() if ngay else "NAN"

# Tìm giá theo m²
info_div3 = page_source.find('div', class_='re__pr-short-info js__pr-short-info')
value_text_area = "NAN"
if info_div3:
    all_items1 = info_div3.find_all('div', class_='re__pr-short-info-item js__pr-short-info-item')
    if all_items1:
        span = all_items1[0].find('span', class_='ext')
        value_text_area = span.get_text().strip() if span else "NAN"

# Tìm các thông tin khác
items = page_source.find_all('div', class_='re__pr-other-info-display')
info_fields = {'Khoảng giá': 'NAN', 'Diện tích': 'NAN','Số tầng': 'NAN', 'Số phòng ngủ': 'NAN', 'Số phòng tắm, vệ sinh': 'NAN','Pháp lý': 'NAN', 'Nội thất': 'NAN'}
for item in items:
    item1 = item.find_all('div', class_="re__pr-specs-content-item")
    for item2 in item1:
        title = item2.find('span', class_="re__pr-specs-content-item-title")
        value = item2.find('span', class_="re__pr-specs-content-item-value")
        if title and value:
            title_text = title.text.strip()
            value_text = value.text.strip()
            if title_text in info_fields:
                info_fields[title_text] = value_text
# Tìm quận
info_div4 = page_source.find('div', class_='re__breadcrumb js__breadcrumb js__ob-breadcrumb')
value_text_district = "NAN"
if info_div4:
    all_items3 = info_div4.find_all('a', class_='re__link-se')
    item1 = all_items3[2] if len(all_items3) > 2 else None
    value_text_district = item1.get_text().strip() if item1 else "NAN"
# Mô tả
desc_div = page_source.select_one('.js__pr-description')
description = desc_div.get_text(separator='\n', strip=True) if desc_div else "NAN"

headers = [
    'Mã code', 'Tiêu đề', 'Ngày đăng', 'Quận','Địa chỉ','Tổng giá', 'Giá theo m²','Diện tích', 'Số tầng',
    'Số phòng ngủ', 'Số phòng tắm, vệ sinh', 'Pháp lý', 'Nội thất', 'Mô tả'
]
List = [
        value_text_code, value_text_title, value_text_date, value_text_district, address_text,  
        info_fields['Khoảng giá'],value_text_area, info_fields['Diện tích'], info_fields['Số tầng'],
        info_fields['Số phòng ngủ'], info_fields['Số phòng tắm, vệ sinh'],
        info_fields['Pháp lý'], info_fields['Nội thất'], description
    ]
result = dict(zip(headers, List))


for k, v in result.items():
    print(f"{k}: {v}")

Mã code: 44964531
Tiêu đề: Nhà mặt phố tại đường Trần Quốc Vượng
Ngày đăng: 02/05/2026
Quận: Cầu Giấy
Địa chỉ: (Phường Cầu Giấy, Hà Nội mới)
Tổng giá: 69,9 tỷ
Giá theo m²: ~665,71 triệu/m²
Diện tích: 105 m²
Số tầng: 8 tầng
Số phòng ngủ: 34 phòng
Số phòng tắm, vệ sinh: 35 phòng
Pháp lý: Sổ đỏ/ Sổ hồng
Nội thất: Đầy đủ
Mô tả: Khách sạn mặt phố - lô góc 3 thoáng - vỉa hè - kinh doanh đỉnh - trung tâm Cầu Giấy.
- Vị trí mặt phố Trần Quốc Vượng, khu lõi trung tâm Cầu Giấy, tuyến phố chuyên kinh doanh khách sạn và căn hộ dịch vụ.
- Lô góc 3 mặt thoáng vĩnh viễn, mặt tiền rộng 10.5M, vỉa hè lớn, ô tô dừng đỗ ngày đêm.
- Diện tích xây dựng mỗi tầng 131m², xây dựng 8 tầng thang máy, thiết kế chuẩn mô hình khách sạn - CHDV.
Công năng khai thác:
- Tầng 1: Lễ tân khách sạn, kinh doanh cafe, khu để xe.
- Tầng 2 - 7: Mỗi tầng 5 phòng khép kín, full nội thất.
- Tầng 8: 4 phòng khép kín + 1 phòng nhân viên.
- Tổng cộng 34 phòng khai thác.
Dòng tiền:
- Tổng dòng tiền thực tế ~330tr/tháng (~4 tỷ/năm).
-

In [98]:
results_list = []

driver.set_page_load_timeout(30)

for i in all_urls:
    try:
        driver.get(i)

        WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.TAG_NAME, "body"))
        )

        page_source = BeautifulSoup(driver.page_source, 'html.parser')

        # Mã code
        info_div1 = page_source.find('div', class_='re__pr-short-info re__pr-config js__pr-config')
        value_text_code = "NAN"
        if info_div1:
            all_items = info_div1.find_all('div', class_='re__pr-short-info-item js__pr-config-item')
            if len(all_items) > 3:
                item = all_items[3]
                value = item.find('span', class_='value')
                if value:
                    value_text_code = value.get_text().strip()

        # Tiêu đề
        info_div_title = page_source.find('div', class_='re__breadcrumb js__breadcrumb js__ob-breadcrumb')
        value_text_title = "NAN"
        if info_div_title:
            all_items3 = info_div_title.find_all('a', class_='re__link-se')
            if len(all_items3) > 3:
                value_text_title = all_items3[3].get_text().strip()

        # Phường/Xã 
        parent_div = page_source.find('div', class_='re__ldp-address')
        address_text = "NAN"
        if parent_div:
            address = parent_div.find('span', class_='re__address-line-2')
            if address:
                address_text = address.get_text(strip=True)

        # Ngày đăng
        value_text_date = "NAN"
        if info_div1:
            ngay = info_div1.find('div')
            if ngay:
                v = ngay.find('span', class_='value')
                if v:
                    value_text_date = v.get_text().strip()

        # Giá/m vuông
        info_div3 = page_source.find('div', class_='re__pr-short-info js__pr-short-info')
        value_text_area = "NAN"
        if info_div3:
            all_items1 = info_div3.find_all('div', class_='re__pr-short-info-item js__pr-short-info-item')
            if all_items1:
                span = all_items1[0].find('span', class_='ext')
                if span:
                    value_text_area = span.get_text().strip()

        # Thông tin 
        info_fields = {
            'Khoảng giá': 'NAN',
            'Diện tích': 'NAN',
            'Số tầng': 'NAN',
            'Số phòng ngủ': 'NAN',
            'Số phòng tắm, vệ sinh': 'NAN',
            'Pháp lý': 'NAN',
            'Nội thất': 'NAN'
        }

        items = page_source.find_all('div', class_='re__pr-other-info-display')
        for item in items:
            for item2 in item.find_all('div', class_="re__pr-specs-content-item"):
                title = item2.find('span', class_="re__pr-specs-content-item-title")
                value = item2.find('span', class_="re__pr-specs-content-item-value")
                if title and value:
                    t = title.text.strip()
                    v = value.text.strip()
                    if t in info_fields:
                        info_fields[t] = v

        # Quận/Huyện
        value_text_district = "NAN"
        if info_div_title:
            all_items3 = info_div_title.find_all('a', class_='re__link-se')
            if len(all_items3) > 2:
                value_text_district = all_items3[2].get_text().strip()

        # Mô tả
        desc_div = page_source.select_one('.js__pr-description')
        description = desc_div.get_text(separator='\n', strip=True) if desc_div else "NAN"

        result = {
            'Mã code': value_text_code,
            'Tiêu đề': value_text_title,
            'Ngày đăng': value_text_date,
            'Quận/Huyện': value_text_district,
            'Phường/Xã': address_text,
            'Tổng giá': info_fields['Khoảng giá'],
            'Giá theo m²': value_text_area,
            'Diện tích': info_fields['Diện tích'],
            'Số tầng': info_fields['Số tầng'],
            'Số phòng ngủ': info_fields['Số phòng ngủ'],
            'Số phòng tắm, vệ sinh': info_fields['Số phòng tắm, vệ sinh'],
            'Pháp lý': info_fields['Pháp lý'],
            'Nội thất': info_fields['Nội thất'],
            'Mô tả': description
        }

        results_list.append(result)

    except Exception as e:
        print("Error:", i, e)
        continue


df1 = pd.DataFrame(results_list)
df1

,Mã code,Tiêu đề,Ngày đăng,Quận/Huyện,Phường/Xã,Tổng giá,Giá theo m²,Diện tích,Số tầng,Số phòng ngủ,"Số phòng tắm, vệ sinh",Pháp lý,Nội thất,Mô tả
0,44964531,Nhà mặt phố tại đường Trần Quốc Vượng,02/05/2026,Cầu Giấy,"(Phường Cầu Giấy, Hà Nội mới)","69,9 tỷ","~665,71 triệu/m²",105 m²,8 tầng,34 phòng,35 phòng,Sổ đỏ/ Sổ hồng,Đầy đủ,Khách sạn mặt phố - lô góc 3 thoáng - vỉa hè -...
1,45019463,"Nhà biệt thự, liền kề tại GIA by KITA",17/03/2026,Tây Hồ,"(Phường Phú Thượng, Hà Nội mới)","60,5 tỷ","~403,33 triệu/m²",150 m²,4 tầng,5 phòng,NAN,Có sổ,NAN,Bán biệt thự liền kề vườn Ciputra đi bộ ra siê...
2,45428439,Nhà riêng tại đường Xuân La,15/04/2026,Tây Hồ,"(Phường Tây Hồ, Hà Nội mới)","26,8 tỷ","~402,4 triệu/m²","66,6 m²",8 tầng,12 phòng,12 phòng,Sổ đỏ/ Sổ hồng,Đầy đủ,Với tiêu chí Giữ tiền + tạo dòng tiền đều mỗi ...
3,45348917,Nhà riêng tại đường Thanh Miến,24/04/2026,Đống Đa,"(Phường Văn Miếu-quốc Tử Giám, Hà Nội mới)",14 tỷ,"~353,54 triệu/m²","39,6 m²",5 tầng,3 phòng,NAN,Sổ đỏ/ Sổ hồng,NAN,Siếu hiếm! Bán nhà đẹp - thang máy xịn - 20m r...
4,44092230,Nhà mặt phố tại đường Bưởi,27/04/2026,Ba Đình,"(Phường Ngọc Hà, Hà Nội mới)","11,9 tỷ","~457,69 triệu/m²",26 m²,6 tầng,NAN,NAN,Sổ đỏ/ Sổ hồng,Đầy đủ,Chính chủ bán nhà mặt phố Bưởi trung tâm Quận ...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
962,45411268,Căn hộ chung cư tại Sunshine Riverside,06/05/2026,Tây Hồ,"(Phường Phú Thượng, Hà Nội mới)","9,2 tỷ","~92,93 triệu/m²",99 m²,NAN,3 phòng,2 phòng,Sổ đỏ/ Sổ hồng,Cơ bản,"Sunshine Riverside, Phường Phú Thượng, là căn ..."
963,44809053,"Nhà biệt thự, liền kề tại Vinhomes Ocean Park ...",28/04/2026,Gia Lâm,"(Xã Gia Lâm, Hà Nội mới)",35 tỷ,"~233,33 triệu/m²",150 m²,NAN,NAN,NAN,Có sổ.,NAN,"Mến chào anh chị,\nChính chủ gửi bán biệt thự ..."
964,45677916,Nhà riêng tại đường Khương Trung,06/05/2026,Thanh Xuân,"(Phường Khương Đình, Hà Nội mới)","29,9 tỷ","~266,96 triệu/m²",112 m²,8 tầng,28 phòng,28 phòng,Sổ đỏ/ Sổ hồng,Đầy đủ,"Hiếm! Toà CHDV Khương Trung 112m²*8T, MT 9m, 2..."
965,44209049,Nhà riêng tại đường Ba La,29/04/2026,Hà Đông,"(Phường Phú Lương, Hà Nội mới)","13,3 tỷ","~255,77 triệu/m²",52 m²,4 tầng,4 phòng,3 phòng,Sổ đỏ/ Sổ hồng,Đầy đủ,Cần bán căn nhà đấu giá 4 tầng nằm trong khu M...


In [99]:
df1.to_csv('data.csv', index=False)

In [100]:
df1.shape

(967, 14)

In [115]:
all_urls[336]

'https://batdongsan.com.vn/ban-shophouse-nha-pho-thuong-mai-duong-to-huu-phuong-la-khe-usilk-city/ban-san-chan-e-chung-cu-dt-3317-2m2-so-lau-dai-co-be-boi-pr45550946'

In [118]:
all_urls[842]

'https://batdongsan.com.vn/ban-nha-mat-pho-pho-pho-hue-phuong-hang-bai-1/khach-san-mp-2-ham-8-noi-dt-300m2-mt-10m-520-ty-pr44724431'

In [119]:
df1 = pd.read_csv("data.csv")

In [120]:
lengths = df1['Mô tả'].astype(str).str.len()

min_len = lengths.min()
max_len = lengths.max()

print("Min length:", min_len)
print("Max length:", max_len)

Min length: 103
Max length: 2825


In [ ]:
44833233

In [5]:
all_urls[153]

'https://batdongsan.com.vn/ban-nha-rieng-duong-tran-duy-hung-phuong-trung-hoa-4-7/yen-cau-giay-49m-5-tang-thang-may-o-to-qua-cua-12-5-ty-pr45452788'